In [0]:
%sh
which aws || true
aws --version || true

/bin/bash: line 2: aws: command not found


In [0]:
%sh
set -euo pipefail

# Install inside the current Databricks user's home directory
INSTALL_DIR="$HOME/.local/aws-cli"
BIN_DIR="$HOME/.local/bin"
TMP_DIR="$(mktemp -d)"

# Automatically remove temporary files
trap 'rm -rf "$TMP_DIR"' EXIT

mkdir -p "$BIN_DIR"
cd "$TMP_DIR"

# Detect Databricks compute architecture
case "$(uname -m)" in
    x86_64)
        AWS_CLI_URL="https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip"
        ;;
    aarch64|arm64)
        AWS_CLI_URL="https://awscli.amazonaws.com/awscli-exe-linux-aarch64.zip"
        ;;
    *)
        echo "Unsupported architecture: $(uname -m)"
        exit 1
        ;;
esac

echo "Downloading AWS CLI..."

curl -fsSL "$AWS_CLI_URL" -o awscliv2.zip
unzip -q awscliv2.zip

# Install or update without sudo
if [ -d "$INSTALL_DIR" ]; then
    ./aws/install \
        --bin-dir "$BIN_DIR" \
        --install-dir "$INSTALL_DIR" \
        --update
else
    ./aws/install \
        --bin-dir "$BIN_DIR" \
        --install-dir "$INSTALL_DIR"
fi

export PATH="$BIN_DIR:$PATH"

echo "AWS CLI installed successfully:"
"$BIN_DIR/aws" --version

You can now run: /home/spark-3b1a5105-d714-4fbf-af11-8b/.local/bin/aws --version
AWS CLI installed successfully:
aws-cli/2.35.15 Python/3.14.5 Linux/6.1.170-210.320.amzn2023.aarch64 exe/aarch64.ubuntu.24


Task 1: AG4P-38 — Data Loading and Quality Assessment
For Task 1, I will check:

Number of CSV files,
Total raw data size,
Dataset columns and data types,
Total rows and unique cities,
Starting and ending dates,
Missing values,
Duplicate city-date records,
Record count for every city

In [0]:
DATA_PATH = "s3://raw-data-bronze-weather"

print("Dataset path:", DATA_PATH)

Dataset path: s3://raw-data-bronze-weather


In [0]:
# Recursively search all folders and collect CSV file details

def find_all_files(folder_path):
    collected_files = []

    try:
        items = dbutils.fs.ls(folder_path)

        for item in items:

            # A folder path normally ends with /
            if item.path.endswith("/"):
                collected_files.extend(
                    find_all_files(item.path)
                )
            else:
                collected_files.append(item)

    except Exception as error:
        print(f"Unable to read: {folder_path}")
        print(error)

    return collected_files


all_files = find_all_files(DATA_PATH)

csv_files = [
    file
    for file in all_files
    if file.path.lower().endswith(".csv")
]

print("Total files found:", len(all_files))
print("Total CSV files found:", len(csv_files))

Total files found: 4687
Total CSV files found: 4686


In [0]:
# Create a table containing file name, path and size

file_records = [
    (
        file.name,
        file.path,
        round(file.size / (1024 ** 2), 2)
    )
    for file in csv_files
]

file_inventory_df = spark.createDataFrame(
    file_records,
    [
        "file_name",
        "file_path",
        "size_mb"
    ]
)

display(
    file_inventory_df.orderBy(
        "size_mb",
        ascending=False
    )
)

file_name,file_path,size_mb
Kilakkarai.csv,s3://raw-data-bronze-weather/w_d_2/Kilakkarai.csv,20.03
Kudavasal.csv,s3://raw-data-bronze-weather/w_d_2/Kudavasal.csv,20.02
Puttalam.csv,s3://raw-data-bronze-weather/w_d_3/Puttalam.csv,20.02
Manamodu.csv,s3://raw-data-bronze-weather/w_d_2/Manamodu.csv,20.01
Pulicat.csv,s3://raw-data-bronze-weather/w_d_3/Pulicat.csv,20.01
Jagannadapuram.csv,s3://raw-data-bronze-weather/w_d_2/Jagannadapuram.csv,20.0
Poonamallee.csv,s3://raw-data-bronze-weather/w_d_3/Poonamallee.csv,19.99
Bommayapalaiyam.csv,s3://raw-data-bronze-weather/w_d_1/Bommayapalaiyam.csv,19.98
Kottaikuppam.csv,s3://raw-data-bronze-weather/w_d_2/Kottaikuppam.csv,19.98
Madukkur.csv,s3://raw-data-bronze-weather/w_d_2/Madukkur.csv,19.98


In [0]:
from pyspark.sql import functions as F

file_summary = (
    file_inventory_df
    .agg(
        F.count("*").alias("total_csv_files"),
        F.round(
            F.sum("size_mb"),
            2
        ).alias("total_size_mb"),
        F.round(
            F.avg("size_mb"),
            2
        ).alias("average_file_size_mb"),
        F.round(
            F.min("size_mb"),
            2
        ).alias("smallest_file_mb"),
        F.round(
            F.max("size_mb"),
            2
        ).alias("largest_file_mb")
    )
)

display(file_summary)

total_csv_files,total_size_mb,average_file_size_mb,smallest_file_mb,largest_file_mb
4686,91781.0,19.59,0.17,20.03


In [0]:
# Read one CSV file first to understand the schema

if len(csv_files) == 0:
    raise ValueError(
        "No CSV files were found. Check DATA_PATH."
    )

sample_file_path = csv_files[0].path

print("Sample file:", sample_file_path)

sample_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(sample_file_path)
)

display(sample_df.limit(10))

Sample file: s3://raw-data-bronze-weather/Weather_Data_Scraping_and_Analysis/weather.csv


_c0,city,lat,lng
0,Delhi,28.61,77.23
1,Mumbai,19.0761,72.8775
2,Kolkata,22.5675,88.37
3,Bangalore,12.9789,77.5917
4,Chennai,13.0825,80.275
5,Hyderabad,17.385,78.4867
6,Pune,18.5203,73.8567
7,Ahmedabad,23.03,72.58
8,Surat,21.1702,72.8311
9,Prayagraj,25.4358,81.8464


In [0]:
# Check column names and detected data types

sample_df.printSchema()

print("Number of columns:", len(sample_df.columns))
print("Column names:")
print(sample_df.columns)

root
 |-- _c0: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lng: double (nullable = true)

Number of columns: 4
Column names:
['_c0', 'city', 'lat', 'lng']


In [0]:
# Use the sample schema to load all CSV files consistently
# Skip the first file (weather.csv) which is metadata, use second file for schema

if len(csv_files) < 2:
    raise ValueError("Not enough CSV files found")

# Get schema from a weather data file (not the metadata file)
weather_data_file = csv_files[1].path
weather_schema_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(weather_data_file)
)
weather_schema = weather_schema_df.schema

# Load only weather data files (skip weather.csv which is file 0)
weather_data_paths = [f.path for f in csv_files[1:]]

raw_weather_df = (
    spark.read
    .schema(weather_schema)
    .option("header", True)
    .csv(weather_data_paths)
)

print("All weather data CSV files loaded successfully.")

All weather data CSV files loaded successfully.


In [0]:
display(raw_weather_df.limit(10))

_c0,city,lat,lng
0,2010-01-01 00:00:00+00:00,22.6805,87.13881
1,2010-01-01 01:00:00+00:00,22.4305,87.656166
2,2010-01-01 02:00:00+00:00,22.880499,86.35467
3,2010-01-01 03:00:00+00:00,24.280499,81.09825
4,2010-01-01 04:00:00+00:00,25.8305,75.7816
5,2010-01-01 05:00:00+00:00,26.9805,71.027664
6,2010-01-01 06:00:00+00:00,27.530499,67.934944
7,2010-01-01 07:00:00+00:00,27.880499,66.5596
8,2010-01-01 08:00:00+00:00,27.880499,66.355515
9,2010-01-01 09:00:00+00:00,28.1805,64.80565


In [0]:
# Convert column names into lowercase and replace spaces with underscores

clean_column_names = [
    column.strip()
    .lower()
    .replace(" ", "_")
    for column in raw_weather_df.columns
]

weather_df = raw_weather_df.toDF(
    *clean_column_names
)

print("Standardized columns:")
print(weather_df.columns)

Standardized columns:
['_c0', 'date', 'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'snow_depth', 'pressure_msl', 'surface_pressure', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'wind_speed_10m', 'wind_speed_100m', 'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m']


In [0]:
from pyspark.sql import functions as F

# Date is already a timestamp, no conversion needed
# Note: city column doesn't exist in weather data files

weather_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- dew_point_2m: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- snow_depth: double (nullable = true)
 |-- pressure_msl: double (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- cloud_cover: double (nullable = true)
 |-- cloud_cover_low: double (nullable = true)
 |-- cloud_cover_mid: double (nullable = true)
 |-- cloud_cover_high: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_speed_100m: double (nullable = true)
 |-- wind_direction_10m: double (nullable = true)
 |-- wind_direction_100m: double (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)



In [0]:
# Calculate total rows and available date range
# Note: city column doesn't exist in weather data files (city info is in file names)

dataset_overview = (
    weather_df
    .agg(
        F.count("*").alias("total_records"),
        F.min("date").alias("starting_date"),
        F.max("date").alias("ending_date")
    )
)

display(dataset_overview)

total_records,starting_date,ending_date
580640160,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z


In [0]:
# Check whether city names are null, empty or only spaces
# Note: weather_df does not have a 'city' column

blank_city_summary = spark.createDataFrame(
    [],
    schema="records_with_blank_city long"
)

display(blank_city_summary)

records_with_blank_city


In [0]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

# Cities selected from different climate zones
sample_cities = [
    "Pune",
    "Mumbai",
    "Nagpur",
    "Delhi",
    "Jaipur",
    "Ahmedabad",
    "Kolkata",
    "Chennai",
    "Bengaluru",
    "Hyderabad",
    "Kochi",
    "Bhopal",
    "Guwahati",
    "Shimla",
    "Srinagar"
]

# Create a lowercase city-to-file mapping
city_file_map = {
    file_path.split("/")[-1]
    .replace(".csv", "")
    .strip()
    .lower(): file_path
    for file_path in weather_data_paths
}

# Store city name together with its correct file path
selected_city_files = [
    (city, city_file_map.get(city.lower()))
    for city in sample_cities
]

# Show cities whose files were not found
missing_city_files = [
    city
    for city, file_path in selected_city_files
    if file_path is None
]

# Keep only available city files
available_city_files = [
    (city, file_path)
    for city, file_path in selected_city_files
    if file_path is not None
]

print("Requested cities:", len(sample_cities))
print("Available city files:", len(available_city_files))
print("Missing city files:", missing_city_files)

Requested cities: 15
Available city files: 12
Missing city files: ['Bengaluru', 'Shimla', 'Srinagar']


In [0]:
missing_value_results = []

for city, file_path in available_city_files:

    city_df = (
        spark.read
        .schema(weather_schema)
        .option("header", True)
        .csv(file_path)
    )

    # Create one missing-value calculation for every column
    missing_expressions = []

    for field in city_df.schema.fields:

        column_name = field.name

        # Check null values
        missing_condition = F.col(column_name).isNull()

        # Also check blank values in string columns
        if field.dataType.simpleString() == "string":
            missing_condition = (
                missing_condition
                | (F.trim(F.col(column_name)) == "")
            )

        missing_expressions.append(
            F.sum(
                F.when(
                    missing_condition,
                    1
                ).otherwise(0)
            ).alias(column_name)
        )

    missing_row = (
        city_df
        .agg(*missing_expressions)
        .first()
    )

    # Store only columns having missing values
    for column_name in city_df.columns:

        missing_count = missing_row[column_name]

        if missing_count > 0:
            missing_value_results.append(
                (
                    city,
                    column_name,
                    int(missing_count)
                )
            )

if missing_value_results:

    missing_values_df = spark.createDataFrame(
        missing_value_results,
        [
            "city",
            "column_name",
            "missing_value_count"
        ]
    )

    display(
        missing_values_df.orderBy(
            F.col("missing_value_count").desc()
        )
    )

else:
    print(
        "No missing values found in the 15 selected city files."
    )

No missing values found in the 15 selected city files.


In [0]:
# Sample 15 specified city files and check for duplicate date records in each

sample_cities = [
    "Pune",
    "Mumbai",
    "Nagpur",
    "Delhi",
    "Jaipur",
    "Ahmedabad",
    "Kolkata",
    "Chennai",
    "Bengaluru",
    "Hyderabad",
    "Kochi",
    "Bhopal",
    "Guwahati",
    "Shimla",
    "Srinagar"
]

# Map city names to file paths
city_file_map = {
    file_path.split("/")[-1].replace(".csv", ""): file_path
    for file_path in weather_data_paths
}

sample_city_files = [
    city_file_map[city]
    for city in sample_cities
    if city in city_file_map
]

duplicate_counts = []

for city, file_path in zip(sample_cities, sample_city_files):
    city_df = (
        spark.read
        .schema(weather_schema)
        .option("header", True)
        .csv(file_path)
    )
    dup_count = (
        city_df
        .groupBy("date")
        .count()
        .filter("count > 1")
        .count()
    )
    duplicate_counts.append((city, dup_count))

duplicates_df = spark.createDataFrame(
    duplicate_counts,
    ["city", "duplicate_date_count"]
)

display(duplicates_df.orderBy("duplicate_date_count", ascending=False))

city,duplicate_date_count
Pune,0
Mumbai,0
Nagpur,0
Delhi,0
Jaipur,0
Ahmedabad,0
Kolkata,0
Chennai,0
Bengaluru,0
Hyderabad,0


In [0]:
city_coverage_results = []

for city, file_path in available_city_files:

    city_df = (
        spark.read
        .schema(weather_schema)
        .option("header", True)
        .csv(file_path)
        .select(
            F.to_timestamp("date").alias(
                "observation_time"
            )
        )
    )

    coverage_row = (
        city_df
        .agg(
            F.count("*").alias(
                "total_records"
            ),
            F.countDistinct(
                "observation_time"
            ).alias(
                "unique_hourly_records"
            ),
            F.min(
                "observation_time"
            ).alias(
                "first_available_date"
            ),
            F.max(
                "observation_time"
            ).alias(
                "last_available_date"
            )
        )
        .first()
    )

    first_date = coverage_row[
        "first_available_date"
    ]

    last_date = coverage_row[
        "last_available_date"
    ]

    unique_hourly_records = coverage_row[
        "unique_hourly_records"
    ]

    # Calculate expected hourly records
    if first_date is not None and last_date is not None:

        expected_hourly_records = int(
            (
                last_date - first_date
            ).total_seconds() / 3600
        ) + 1

        missing_hourly_records = max(
            expected_hourly_records
            - unique_hourly_records,
            0
        )

        coverage_percentage = round(
            unique_hourly_records
            / expected_hourly_records
            * 100,
            2
        )

    else:
        expected_hourly_records = 0
        missing_hourly_records = 0
        coverage_percentage = 0.0

    city_coverage_results.append(
        (
            city,
            int(coverage_row["total_records"]),
            int(unique_hourly_records),
            first_date,
            last_date,
            expected_hourly_records,
            missing_hourly_records,
            coverage_percentage
        )
    )

city_coverage_df = spark.createDataFrame(
    city_coverage_results,
    [
        "city",
        "total_records",
        "unique_hourly_records",
        "first_available_date",
        "last_available_date",
        "expected_hourly_records",
        "missing_hourly_records",
        "coverage_percentage"
    ]
)

display(
    city_coverage_df.orderBy(
        F.col("coverage_percentage").asc()
    )
)

city,total_records,unique_hourly_records,first_available_date,last_available_date,expected_hourly_records,missing_hourly_records,coverage_percentage
Pune,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Mumbai,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Nagpur,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Delhi,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Jaipur,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Ahmedabad,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Kolkata,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Chennai,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Hyderabad,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
Kochi,123936,123936,2010-01-01T00:00:00.000Z,2024-02-20T23:00:00.000Z,123936,0,100.0
